This notebook serves as a place containing my work for homework 4

# Question 1.)

 Build three models for FBS team evaluation: an SRS model of score differential only, an opponent adjusted game control model, and an opponent adjusted average win probability (awp) model.  Give the top 10 and botton 10 teams for each season from 2021 to 2025.

In [18]:
# import neccesary libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns
from IPython.display import display, Markdown
import sportsdataverse as sdv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

# SRS model (score differential only)

In [6]:
# define the season range
seasons = range(2021, 2026)
games_list = []

for season in seasons:
    url = f"https://raw.githubusercontent.com/sportsdataverse/cfbfastR-data/main/schedules/csv/cfb_schedules_{season}.csv"
    df_season = pd.read_csv(url)
    games_list.append(df_season)

raw_games = pd.concat(games_list, ignore_index=True)

# Clean and filter for completed games with valid scores
games_clean = raw_games[
    (raw_games['completed'] == True) &
    (raw_games['home_points'].notna()) &
    (raw_games['away_division']=='fbs') &
    (raw_games['away_points'].notna()) &
    (raw_games['season_type'].isin(['regular']))
][['season', 'week', 'home_team', 'away_team', 'home_points', 'away_points']].sort_values(['season', 'week']).reset_index(drop=True)

In [7]:
# Make an SRS function
def calc_srs_scratch(df):
    # Reset index to guarantee alignment with 0..n_games range
    df = df.reset_index(drop=True)
    
    teams = np.unique(np.concatenate([df['home_team'].unique(), df['away_team'].unique()]))
    n_games = len(df)
    
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, row in df.iterrows():
        X.loc[i, row['home_team']] = 1.0
        X.loc[i, row['away_team']] = -1.0
        
    margin = df['home_points'] - df['away_points']
    X = sm.add_constant(X)
    fit = sm.OLS(margin, X).fit()
    srs_vec = fit.params.fillna(0)
    srs_centered = srs_vec - srs_vec.mean()
    
    return pd.DataFrame({
        'team': srs_centered.index,
        'SRS': np.round(srs_centered.values, 2)
    })

In [8]:
# calculate SRS for each season
def compute_season_ratings(season_df):
    year = season_df['season'].iloc[0]
    srs_df = calc_srs_scratch(season_df)
    srs_df['season'] = year
    return srs_df.sort_values(by='SRS', ascending=False).reset_index(drop=True)

# Execute loop across split season datasets
ratings_list = []
for year, season_df in games_clean.groupby('season'):
    ratings_list.append(compute_season_ratings(season_df))

ratings_by_season = pd.concat(ratings_list, ignore_index=True)

C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3867389675.py:17: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3867389675.py:17: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3867389675.py:17: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3867389675.py:17: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3867389675.py:17: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely

In [9]:
# List top 10 teams by SRS for each season
for yr in sorted(ratings_by_season['season'].unique()):
    top_rankings = (
        ratings_by_season[ratings_by_season['season'] == yr]
        .sort_values(by='SRS', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    top_rankings['Rank'] = top_rankings.index + 1
    
    top_rankings = top_rankings[[
        'Rank', 'team', 'SRS'
    ]].rename(columns={
        'team': 'Team',
        'SRS': 'SRS Rating',
    })
    
    display(Markdown(f"### Top 10 Teams by SRS for Season: {yr}"))
    display(Markdown(top_rankings.to_markdown(index=False)))

### Top 10 Teams by SRS for Season: 2021

|   Rank | Team           |   SRS Rating |
|-------:|:---------------|-------------:|
|      1 | Georgia        |        35.58 |
|      2 | Alabama        |        29.23 |
|      3 | Ohio State     |        28.62 |
|      4 | Michigan       |        25.77 |
|      5 | Notre Dame     |        21.67 |
|      6 | Oklahoma State |        19.92 |
|      7 | Utah           |        19.82 |
|      8 | Cincinnati     |        19.35 |
|      9 | Iowa State     |        17.16 |
|     10 | Wisconsin      |        16.57 |

### Top 10 Teams by SRS for Season: 2022

|   Rank | Team         |   SRS Rating |
|-------:|:-------------|-------------:|
|      1 | Georgia      |        33.86 |
|      2 | Tennessee    |        30.29 |
|      3 | Alabama      |        29.76 |
|      4 | Ohio State   |        29.22 |
|      5 | Michigan     |        28.03 |
|      6 | Texas        |        25.14 |
|      7 | Kansas State |        24.52 |
|      8 | TCU          |        23.58 |
|      9 | Penn State   |        22.47 |
|     10 | Utah         |        20.41 |

### Top 10 Teams by SRS for Season: 2023

|   Rank | Team          |   SRS Rating |
|-------:|:--------------|-------------:|
|      1 | Michigan      |        30.28 |
|      2 | Oregon        |        29.97 |
|      3 | Ohio State    |        28.46 |
|      4 | Penn State    |        27.53 |
|      5 | Oklahoma      |        25.83 |
|      6 | Texas         |        25.66 |
|      7 | Georgia       |        24.26 |
|      8 | Notre Dame    |        23.39 |
|      9 | Kansas State  |        22.73 |
|     10 | Florida State |        22.38 |

### Top 10 Teams by SRS for Season: 2024

|   Rank | Team           |   SRS Rating |
|-------:|:---------------|-------------:|
|      1 | Notre Dame     |        27.95 |
|      2 | Texas          |        27.33 |
|      3 | Ohio State     |        27.14 |
|      4 | Alabama        |        25.71 |
|      5 | Ole Miss       |        24.5  |
|      6 | Indiana        |        23.38 |
|      7 | Georgia        |        22.95 |
|      8 | Tennessee      |        22.64 |
|      9 | Oregon         |        22.63 |
|     10 | South Carolina |        21.74 |

### Top 10 Teams by SRS for Season: 2025

|   Rank | Team       |   SRS Rating |
|-------:|:-----------|-------------:|
|      1 | Indiana    |        37.28 |
|      2 | Ohio State |        32.5  |
|      3 | Texas Tech |        31.9  |
|      4 | Oregon     |        30.13 |
|      5 | Notre Dame |        29.44 |
|      6 | Utah       |        25.56 |
|      7 | Miami      |        24.63 |
|      8 | Georgia    |        22.46 |
|      9 | USC        |        21.54 |
|     10 | Iowa       |        21.02 |

In [10]:
# List bottom 10 teams by SRS for each season
for yr in sorted(ratings_by_season['season'].unique()):
    bottom_rankings = (
        ratings_by_season[ratings_by_season['season'] == yr]
        .sort_values(by='SRS', ascending=True)
        .head(10)
        .reset_index(drop=True)
    )
    bottom_rankings['Rank'] = bottom_rankings.index + 1
    
    bottom_rankings = bottom_rankings[[
        'Rank', 'team', 'SRS'
    ]].rename(columns={
        'team': 'Team',
        'SRS': 'SRS Rating',
    })
    
    display(Markdown(f"### Bottom 10 Teams by SRS for Season: {yr}"))
    display(Markdown(bottom_rankings.to_markdown(index=False)))

### Bottom 10 Teams by SRS for Season: 2021

|   Rank | Team                  |   SRS Rating |
|-------:|:----------------------|-------------:|
|      1 | Florida International |       -33.09 |
|      2 | Jacksonville State    |       -32.51 |
|      3 | Massachusetts         |       -30.72 |
|      4 | Akron                 |       -29.86 |
|      5 | Temple                |       -28.79 |
|      6 | UConn                 |       -28.02 |
|      7 | Southern Miss         |       -22.21 |
|      8 | New Mexico State      |       -21.6  |
|      9 | Arkansas State        |       -21.33 |
|     10 | Duke                  |       -20.71 |

### Bottom 10 Teams by SRS for Season: 2022

|   Rank | Team                  |   SRS Rating |
|-------:|:----------------------|-------------:|
|      1 | Florida International |       -32.56 |
|      2 | Massachusetts         |       -29.01 |
|      3 | New Mexico            |       -24.59 |
|      4 | Charlotte             |       -23.77 |
|      5 | Hawai'i               |       -23.42 |
|      6 | Louisiana Tech        |       -20.61 |
|      7 | New Mexico State      |       -20.21 |
|      8 | Nevada                |       -19.8  |
|      9 | Colorado              |       -19.49 |
|     10 | Akron                 |       -18.54 |

### Bottom 10 Teams by SRS for Season: 2023

|   Rank | Team                  |   SRS Rating |
|-------:|:----------------------|-------------:|
|      1 | Kent State            |       -32.03 |
|      2 | Temple                |       -25.42 |
|      3 | Akron                 |       -24.01 |
|      4 | UL Monroe             |       -21.87 |
|      5 | Florida International |       -20.77 |
|      6 | Massachusetts         |       -20.18 |
|      7 | Eastern Michigan      |       -19.15 |
|      8 | Charlotte             |       -18.8  |
|      9 | Nevada                |       -18.68 |
|     10 | Southern Miss         |       -17.91 |

### Bottom 10 Teams by SRS for Season: 2024

|   Rank | Team             |   SRS Rating |
|-------:|:-----------------|-------------:|
|      1 | Kent State       |       -34.84 |
|      2 | Tulsa            |       -32.78 |
|      3 | Southern Miss    |       -29    |
|      4 | Middle Tennessee |       -26.85 |
|      5 | New Mexico State |       -26.72 |
|      6 | Kennesaw State   |       -26.32 |
|      7 | UTEP             |       -23.2  |
|      8 | Temple           |       -21.88 |
|      9 | Ball State       |       -21.04 |
|     10 | Central Michigan |       -20.98 |

### Bottom 10 Teams by SRS for Season: 2025

|   Rank | Team          |   SRS Rating |
|-------:|:--------------|-------------:|
|      1 | Massachusetts |       -37.85 |
|      2 | Sam Houston   |       -28.88 |
|      3 | UL Monroe     |       -25.6  |
|      4 | Charlotte     |       -24.65 |
|      5 | Georgia State |       -23.24 |
|      6 | Akron         |       -22.99 |
|      7 | Ball State    |       -22.58 |
|      8 | UTEP          |       -20.7  |
|      9 | Kent State    |       -20.29 |
|     10 | Rice          |       -19.27 |

# Opponent-adjusted game control model

Get the data.

In [ ]:
# Import college football data from 2021 to 2025
cfb_data = sdv.cfb.load_cfb_pbp([2021, 2022, 2023, 2024, 2025], return_as_pandas=False)

# Convert the polar files to pandas dataframes
cfb_data = cfb_data.to_pandas(use_pyarrow_extension_array=False)

In [ ]:
# Create a list of all FBS schools
fbs_teams = ["Boston College", "California", "Clemson", "Duke", "Florida State",
    "Georgia Tech", "Louisville", "Miami", "NC State", "North Carolina",
    "Pittsburgh", "SMU", "Stanford", "Syracuse", "Virginia",
    "Virginia Tech", "Wake Forest", "Illinois", "Indiana", "Iowa", "Maryland", "Michigan",
    "Michigan State", "Minnesota", "Nebraska", "Northwestern", "Ohio State",
    "Oregon", "Penn State", "Purdue", "Rutgers", "UCLA",
    "USC", "Washington", "Wisconsin", "Arizona", "Arizona State", "Baylor", "BYU", "Cincinnati",
    "Colorado", "Houston", "Iowa State", "Kansas", "Kansas State",
    "Oklahoma State", "TCU", "Texas Tech", "UCF", "Utah",
    "West Virginia", "Alabama", "Arkansas", "Auburn", "Florida", "Georgia",
    "Kentucky", "LSU", "Mississippi State", "Missouri", "Oklahoma",
    "Ole Miss", "South Carolina", "Tennessee", "Texas", "Texas A&M",
    "Vanderbilt", "Army", "Charlotte", "East Carolina", "Florida Atlantic", "Memphis",
    "Navy", "North Texas", "Rice", "Temple", "Tulane",
    "Tulsa", "UAB", "South Florida", "UTSA", "Boise State", "Colorado State", "Fresno State",
    "Oregon State", "San Diego State", "Texas State", "Utah State", "Washington State",
    "Air Force", "Hawai'i", "Nevada", "New Mexico", "North Dakota State",
    "Northern Illinois", "San José State", "UNLV", "UTEP", "Wyoming",
    "Akron", "Ball State", "Bowling Green", "Buffalo", "Central Michigan",
    "Eastern Michigan", "Kent State", "Miami (OH)", "Ohio", "Sacramento State",
    "Toledo", "Massachusetts", "Western Michigan",
    "Delaware", "Florida International", "Jacksonville State", "Kennesaw State", "Liberty",
    "Middle Tennessee", "Missouri State", "New Mexico State", "Sam Houston", "Western Kentucky",
    "Appalachian State", "Arkansas State", "Coastal Carolina", "Georgia Southern", "Georgia State",
    "James Madison", "Louisiana", "Louisiana Tech", "Marshall", "Old Dominion",
    "South Alabama", "Southern Miss", "Troy", "UL Monroe", "Notre Dame", "UConn"
]

# Now, let's filter the data so that it only keeps fbs vs fbs games
# and it only keeps the regular season games and plays that we care about on offense.
cfb_data = cfb_data[(cfb_data['homeTeamName'].isin(fbs_teams)) & (cfb_data['awayTeamName'].isin(fbs_teams))
].copy()

In [ ]:
# Create a dictionary to map the values
school_mapping = {
    "Boston College Eagles": "Boston College",
    "California Golden Bears": "California",
    "Clemson Tigers": "Clemson",
    "Duke Blue Devils": "Duke",
    "Florida State Seminoles": "Florida State",
    "Georgia Tech Yellow Jackets": "Georgia Tech",
    "Louisville Cardinals": "Louisville",
    "Miami Hurricanes": "Miami",
    "NC State Wolfpack": "NC State",
    "North Carolina Tar Heels": "North Carolina",
    "Pittsburgh Panthers": "Pittsburgh",
    "SMU Mustangs": "SMU",
    "Stanford Cardinal": "Stanford",
    "Syracuse Orange": "Syracuse",
    "Virginia Cavaliers": "Virginia",
    "Virginia Tech Hokies": "Virginia Tech",
    "Wake Forest Demon Deacons": "Wake Forest",
    "Illinois Fighting Illini": "Illinois",
    "Indiana Hoosiers": "Indiana",
    "Iowa Hawkeyes": "Iowa",
    "Maryland Terrapins": "Maryland",
    "Michigan Wolverines": "Michigan",
    "Michigan State Spartans": "Michigan State",
    "Minnesota Golden Gophers": "Minnesota",
    "Nebraska Cornhuskers": "Nebraska",
    "Northwestern Wildcats": "Northwestern",
    "Ohio State Buckeyes": "Ohio State",
    "Oregon Ducks": "Oregon",
    "Penn State Nittany Lions": "Penn State",
    "Purdue Boilermakers": "Purdue",
    "Rutgers Scarlet Knights": "Rutgers",
    "UCLA Bruins": "UCLA",
    "USC Trojans": "USC",
    "Washington Huskies": "Washington",
    "Wisconsin Badgers": "Wisconsin",
    "Arizona Wildcats": "Arizona",
    "Arizona State Sun Devils": "Arizona State",
    "Baylor Bears": "Baylor",
    "BYU Cougars": "BYU",
    "Cincinnati Bearcats": "Cincinnati",
    "Colorado Buffaloes": "Colorado",
    "Houston Cougars": "Houston",
    "Iowa State Cyclones": "Iowa State",
    "Kansas Jayhawks": "Kansas",
    "Kansas State Wildcats": "Kansas State",
    "Oklahoma State Cowboys": "Oklahoma State",
    "TCU Horned Frogs": "TCU",
    "Texas Tech Red Raiders": "Texas Tech",
    "UCF Knights": "UCF",
    "Utah Utes": "Utah",
    "West Virginia Mountaineers": "West Virginia",
    "Alabama Crimson Tide": "Alabama",
    "Arkansas Razorbacks": "Arkansas",
    "Auburn Tigers": "Auburn",
    "Florida Gators": "Florida",
    "Georgia Bulldogs": "Georgia",
    "Kentucky Wildcats": "Kentucky",
    "LSU Tigers": "LSU",
    "Mississippi State Bulldogs": "Mississippi State",
    "Missouri Tigers": "Missouri",
    "Oklahoma Sooners": "Oklahoma",
    "Ole Miss Rebels": "Ole Miss",
    "South Carolina Gamecocks": "South Carolina",
    "Tennessee Volunteers": "Tennessee",
    "Texas Longhorns": "Texas",
    "Texas A&M Aggies": "Texas A&M",
    "Vanderbilt Commodores": "Vanderbilt",
    "Army Black Knights": "Army",
    "Charlotte 49ers": "Charlotte",
    "East Carolina Pirates": "East Carolina",
    "Florida Atlantic Owls": "Florida Atlantic",
    "Memphis Tigers": "Memphis",
    "Navy Midshipmen": "Navy",
    "North Texas Mean Green": "North Texas",
    "Rice Owls": "Rice",
    "Temple Owls": "Temple",
    "Tulane Green Wave": "Tulane",
    "Tulsa Golden Hurricane": "Tulsa",
    "UAB Blazers": "UAB",
    "South Florida Bulls": "South Florida",
    "UTSA Roadrunners": "UTSA",
    "Boise State Broncos": "Boise State",
    "Colorado State Rams": "Colorado State",
    "Fresno State Bulldogs": "Fresno State",
    "Oregon State Beavers": "Oregon State",
    "San Diego State Aztecs": "San Diego State",
    "Texas State Bobcats": "Texas State",
    "Utah State Aggies": "Utah State",
    "Washington State Cougars": "Washington State",
    "Air Force Falcons": "Air Force",
    "Hawai'i Rainbow Warriors": "Hawai'i",
    "Nevada Wolf Pack": "Nevada",
    "New Mexico Lobos": "New Mexico",
    "North Dakota State Bison": "North Dakota State",
    "Northern Illinois Huskies": "Northern Illinois",
    "San José State Spartans": "San José State",
    "UNLV Rebels": "UNLV",
    "UTEP Miners": "UTEP",
    "Wyoming Cowboys": "Wyoming",
    "Akron Zips": "Akron",
    "Ball State Cardinals": "Ball State",
    "Bowling Green Falcons": "Bowling Green",
    "Buffalo Bulls": "Buffalo",
    "Central Michigan Chippewas": "Central Michigan",
    "Eastern Michigan Eagles": "Eastern Michigan",
    "Kent State Golden Flashes": "Kent State",
    "Miami (OH) RedHawks": "Miami (OH)",
    "Ohio Bobcats": "Ohio",
    "Sacramento State Hornets": "Sacramento State",
    "Toledo Rockets": "Toledo",
    "Massachusetts Minutemen": "Massachusetts",
    "Western Michigan Broncos": "Western Michigan",
    "Delaware Blue Hens": "Delaware",
    "Florida International Panthers": "Florida International",
    "Jacksonville State Gamecocks": "Jacksonville State",
    "Kennesaw State Owls": "Kennesaw State",
    "Liberty Flames": "Liberty",
    "Middle Tennessee Blue Raiders": "Middle Tennessee",
    "Missouri State Bears": "Missouri State",
    "New Mexico State Aggies": "New Mexico State",
    "Sam Houston Bearkats": "Sam Houston",
    "Western Kentucky Hilltoppers": "Western Kentucky",
    "Appalachian State Mountaineers": "Appalachian State",
    "Arkansas State Red Wolves": "Arkansas State",
    "Coastal Carolina Chanticleers": "Coastal Carolina",
    "Georgia Southern Eagles": "Georgia Southern",
    "Georgia State Panthers": "Georgia State",
    "James Madison Dukes": "James Madison",
    "Louisiana Ragin' Cajuns": "Louisiana",
    "Louisiana Tech Bulldogs": "Louisiana Tech",
    "Marshall Thundering Herd": "Marshall",
    "Old Dominion Monarchs": "Old Dominion",
    "South Alabama Jaguars": "South Alabama",
    "Southern Miss Golden Eagles": "Southern Miss",
    "Troy Trojans": "Troy",
    "UL Monroe Warhawks": "UL Monroe",
    "Notre Dame Fighting Irish": "Notre Dame",
    "UConn Huskies": "UConn",
}

# Map the values to replace the values in pos_team and def_pos_team
cfb_data['pos_team'] = cfb_data['pos_team'].map(school_mapping)
cfb_data['def_pos_team'] = cfb_data['def_pos_team'].map(school_mapping)

# Check that it worked as expected
cfb_data['pos_team'].value_counts()

In [11]:
# make a game seconds remaining stat for the pbp data
cfb_data['game_seconds_remaining'] = np.where(cfb_data['half'] == 1, cfb_data['end.TimeSecsRem'].astype(float) + 1800, cfb_data['end.TimeSecsRem'].astype(float))

In [12]:
# Filter completed plays and compute duration between play events
pbp_clean = cfb_data[
    (cfb_data['seasonType'] == 2) &
    (cfb_data['game_seconds_remaining'].notna()) &
    (cfb_data['homeScore'].notna()) &
    (cfb_data['awayScore'].notna()) &
    (cfb_data['homeTeamName'].notna()) &
    (cfb_data['awayTeamName'].notna())
].copy()

pbp_clean = pbp_clean.sort_values(['season', 'game_id', 'game_seconds_remaining'], ascending=[True, True, False])

pbp_clean['home_lead'] = pbp_clean['homeScore'] - pbp_clean['awayScore']

# Elapsed seconds from previous play state
prev_seconds = pbp_clean.groupby('game_id')['game_seconds_remaining'].shift(1).fillna(3600)
pbp_clean['play_duration'] = prev_seconds - pbp_clean['game_seconds_remaining']
pbp_clean['play_duration'] = np.maximum(0, pbp_clean['play_duration'])

In [13]:
# Calculate time-weighted score differential per game for home and away teams
def calc_game_gc(group):
    total_time = group['play_duration'].sum()
    denom = total_time if total_time > 0 else 1.0
    home_gc = (group['home_lead'] * group['play_duration']).sum() / denom
    return pd.Series({
        'total_time': total_time,
        'home_gc': home_gc,
        'away_gc': -home_gc
    })

game_control_by_game = pbp_clean.groupby(['season', 'game_id', 'homeTeamName', 'awayTeamName'], group_keys=False).apply(calc_game_gc).reset_index()

# Reshape data into long team-game format
home_df = game_control_by_game[['season', 'game_id', 'homeTeamName', 'home_gc']].rename(columns={'homeTeamName': 'team', 'home_gc': 'game_control'})
away_df = game_control_by_game[['season', 'game_id', 'awayTeamName', 'away_gc']].rename(columns={'awayTeamName': 'team', 'away_gc': 'game_control'})

team_games = pd.concat([home_df, away_df], ignore_index=True)

# Aggregate across each season
season_game_control = team_games.groupby(['season', 'team']).agg(
    games_played=('game_control', 'count'),
    avg_game_control=('game_control', lambda x: np.round(x.mean(), 2))
).reset_index().sort_values(['season', 'avg_game_control'], ascending=[True, False])

In [14]:
def calc_adj_gc_scratch(df):
    teams = np.unique(np.concatenate([df['homeTeamName'].unique(), df['awayTeamName'].unique()]))
    n_games = len(df)
    season = df['season'].iloc[0]
    
    # Construct design matrix: Home (+1), Away (-1)
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, (_, row) in enumerate(df.iterrows()):
        X.loc[i, row['homeTeamName']] = 1.0
        X.loc[i, row['awayTeamName']] = -1.0
        
    margin = df['home_gc'].values
    
    # Add intercept column (column of ones)
    X_design = sm.add_constant(X)
    #X_design=X   
    # Fit OLS model
    fit = sm.OLS(margin, X_design).fit()
    srs_vec = fit.params.fillna(0)
    srs_vec = srs_vec.rename(index={'const': '(Intercept)'})
    
    # Center parameters
    srs_centered = srs_vec - srs_vec.mean()
    
    return pd.DataFrame({
        'team': srs_centered.index,
        'season': season,
        'Adj_GC': np.round(srs_centered.values, 2)
    })

In [15]:
ratings_list = []
for yr, season_df in game_control_by_game.groupby('season'):
    ratings_list.append(calc_adj_gc_scratch(season_df))

ratings_by_season = pd.concat(ratings_list, ignore_index=True)

C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3482623808.py:19: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X_design).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3482623808.py:19: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X_design).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3482623808.py:19: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X_design).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3482623808.py:19: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(margin, X_design).fit()
C:\Users\ajhay\AppData\Local\Temp\ipykernel_5716\3482623808.py:19: SingularMatrixWarning: The design matrix is rank-deficient. The model

In [16]:
# get top 10 teams by adjusted game control for each season
for yr in sorted(ratings_by_season['season'].unique()):
    top_10 = (
        ratings_by_season[ratings_by_season['season'] == yr]
        .sort_values(by='Adj_GC', ascending=False).head(10)
        .reset_index(drop=True)
    )
    top_10['Rank'] = top_10.index + 1
    
    top_10 = top_10[['Rank', 'season', 'team', 'Adj_GC']].rename(columns={
        'season': 'season',
        'team': 'Team',
        'Adj_GC': 'Adj_game_control'
    })
    
    display(Markdown(f"### Top 10 Teams by Adjusted Game Control for Season: {yr}"))
    display(Markdown(top_10.to_markdown(index=False)))

### Top 10 Teams by Adjusted Game Control for Season: 2021

|   Rank |   season | Team       |   Adj_game_control |
|-------:|---------:|:-----------|-------------------:|
|      1 |     2021 | Georgia    |              22.12 |
|      2 |     2021 | Alabama    |              18.16 |
|      3 |     2021 | Ohio State |              16.57 |
|      4 |     2021 | Michigan   |              13.98 |
|      5 |     2021 | Ole Miss   |              11.99 |
|      6 |     2021 | Utah       |              11.61 |
|      7 |     2021 | Tennessee  |              11.58 |
|      8 |     2021 | Cincinnati |              11.34 |
|      9 |     2021 | Notre Dame |              10.17 |
|     10 |     2021 | Wisconsin  |              10.15 |

### Top 10 Teams by Adjusted Game Control for Season: 2022

|   Rank |   season | Team         |   Adj_game_control |
|-------:|---------:|:-------------|-------------------:|
|      1 |     2022 | Georgia      |              18.56 |
|      2 |     2022 | Texas        |              17.05 |
|      3 |     2022 | Alabama      |              16.64 |
|      4 |     2022 | Kansas State |              15.81 |
|      5 |     2022 | Ohio State   |              15.62 |
|      6 |     2022 | Tennessee    |              15.31 |
|      7 |     2022 | Utah         |              12.91 |
|      8 |     2022 | Michigan     |              12.62 |
|      9 |     2022 | TCU          |              12.35 |
|     10 |     2022 | Oregon       |              12.35 |

### Top 10 Teams by Adjusted Game Control for Season: 2023

|   Rank |   season | Team         |   Adj_game_control |
|-------:|---------:|:-------------|-------------------:|
|      1 |     2023 | Oregon       |              17.29 |
|      2 |     2023 | Michigan     |              15.85 |
|      3 |     2023 | Oklahoma     |              14.72 |
|      4 |     2023 | Texas        |              14.26 |
|      5 |     2023 | Washington   |              13.1  |
|      6 |     2023 | Ohio State   |              12.84 |
|      7 |     2023 | Georgia      |              12.66 |
|      8 |     2023 | Notre Dame   |              11.64 |
|      9 |     2023 | Kansas State |              11.3  |
|     10 |     2023 | LSU          |              11.3  |

### Top 10 Teams by Adjusted Game Control for Season: 2024

|   Rank |   season | Team          |   Adj_game_control |
|-------:|---------:|:--------------|-------------------:|
|      1 |     2024 | Texas         |              15.15 |
|      2 |     2024 | Alabama       |              14.03 |
|      3 |     2024 | Notre Dame    |              13.99 |
|      4 |     2024 | Ole Miss      |              13.68 |
|      5 |     2024 | Ohio State    |              13.02 |
|      6 |     2024 | Oregon        |              12.87 |
|      7 |     2024 | Indiana       |              11.22 |
|      8 |     2024 | Tennessee     |              11.17 |
|      9 |     2024 | Clemson       |              10.93 |
|     10 |     2024 | Arizona State |              10.3  |

### Top 10 Teams by Adjusted Game Control for Season: 2025

|   Rank |   season | Team       |   Adj_game_control |
|-------:|---------:|:-----------|-------------------:|
|      1 |     2025 | Indiana    |              18.24 |
|      2 |     2025 | Texas Tech |              17.59 |
|      3 |     2025 | Notre Dame |              16.94 |
|      4 |     2025 | Oregon     |              16.79 |
|      5 |     2025 | Ohio State |              15.68 |
|      6 |     2025 | Utah       |              12.84 |
|      7 |     2025 | Miami      |              12.78 |
|      8 |     2025 | Oklahoma   |              11.78 |
|      9 |     2025 | Georgia    |              11.76 |
|     10 |     2025 | Alabama    |              11.09 |

In [17]:
# get bottom 10 teams by adjusted game control for each season
for yr in sorted(ratings_by_season['season'].unique()):
    bottom_10 = (
        ratings_by_season[ratings_by_season['season'] == yr]
        .sort_values(by='Adj_GC', ascending=True).head(10)
        .reset_index(drop=True)
    )
    bottom_10['Rank'] = bottom_10.index + 1
    
    bottom_10 = bottom_10[['Rank', 'season', 'team', 'Adj_GC']].rename(columns={
        'season': 'season',
        'team': 'Team',
        'Adj_GC': 'Adj_game_control'
    })
    
    display(Markdown(f"### Bottom 10 Teams by Adjusted Game Control for Season: {yr}"))
    display(Markdown(bottom_10.to_markdown(index=False)))

### Bottom 10 Teams by Adjusted Game Control for Season: 2021

|   Rank |   season | Team                  |   Adj_game_control |
|-------:|---------:|:----------------------|-------------------:|
|      1 |     2021 | Kennesaw State        |             -23.81 |
|      2 |     2021 | Massachusetts         |             -20.88 |
|      3 |     2021 | Delaware              |             -20.57 |
|      4 |     2021 | UConn                 |             -16.04 |
|      5 |     2021 | Florida International |             -15.77 |
|      6 |     2021 | Temple                |             -15.52 |
|      7 |     2021 | Akron                 |             -14.75 |
|      8 |     2021 | Arkansas State        |             -14.47 |
|      9 |     2021 | New Mexico State      |             -11.53 |
|     10 |     2021 | Southern Miss         |             -11.48 |

### Bottom 10 Teams by Adjusted Game Control for Season: 2022

|   Rank |   season | Team                  |   Adj_game_control |
|-------:|---------:|:----------------------|-------------------:|
|      1 |     2022 | Jacksonville State    |             -32.87 |
|      2 |     2022 | Kennesaw State        |             -20.76 |
|      3 |     2022 | Florida International |             -16.56 |
|      4 |     2022 | Massachusetts         |             -14.78 |
|      5 |     2022 | Hawai'i               |             -14.46 |
|      6 |     2022 | Nevada                |             -13.72 |
|      7 |     2022 | Sam Houston           |             -12.25 |
|      8 |     2022 | New Mexico State      |             -11.61 |
|      9 |     2022 | Colorado State        |             -11.36 |
|     10 |     2022 | Akron                 |             -11.13 |

### Bottom 10 Teams by Adjusted Game Control for Season: 2023

|   Rank |   season | Team                  |   Adj_game_control |
|-------:|---------:|:----------------------|-------------------:|
|      1 |     2023 | Delaware              |             -15.61 |
|      2 |     2023 | Kent State            |             -15.21 |
|      3 |     2023 | Massachusetts         |             -13.31 |
|      4 |     2023 | Temple                |             -12.99 |
|      5 |     2023 | Akron                 |             -12.13 |
|      6 |     2023 | Hawai'i               |             -11.01 |
|      7 |     2023 | UL Monroe             |             -10.88 |
|      8 |     2023 | Florida International |             -10.86 |
|      9 |     2023 | Louisiana Tech        |             -10.83 |
|     10 |     2023 | Eastern Michigan      |             -10.5  |

### Bottom 10 Teams by Adjusted Game Control for Season: 2024

|   Rank |   season | Team             |   Adj_game_control |
|-------:|---------:|:-----------------|-------------------:|
|      1 |     2024 | Kent State       |             -21.53 |
|      2 |     2024 | Tulsa            |             -21.13 |
|      3 |     2024 | Middle Tennessee |             -16.71 |
|      4 |     2024 | New Mexico State |             -15.11 |
|      5 |     2024 | Southern Miss    |             -14.32 |
|      6 |     2024 | Kennesaw State   |             -14.19 |
|      7 |     2024 | UTEP             |             -13.09 |
|      8 |     2024 | Temple           |             -12.39 |
|      9 |     2024 | Central Michigan |             -11.29 |
|     10 |     2024 | Purdue           |             -11.17 |

### Bottom 10 Teams by Adjusted Game Control for Season: 2025

|   Rank |   season | Team             |   Adj_game_control |
|-------:|---------:|:-----------------|-------------------:|
|      1 |     2025 | Massachusetts    |             -19.27 |
|      2 |     2025 | Sam Houston      |             -15.86 |
|      3 |     2025 | Charlotte        |             -14.14 |
|      4 |     2025 | Ball State       |             -13.61 |
|      5 |     2025 | UL Monroe        |             -12.15 |
|      6 |     2025 | Rice             |             -12.11 |
|      7 |     2025 | Akron            |             -12.05 |
|      8 |     2025 | Kent State       |             -11.77 |
|      9 |     2025 | Coastal Carolina |             -11.6  |
|     10 |     2025 | UTEP             |             -11.47 |

# Average Win probability model

add def yds allowed after 
    avg_yds_allowed=('def_yds_play', 'mean')

In [19]:
# Examine the available columns of the play by play data
col_list = cfb_data.columns.tolist()
pd.set_option('display.max_seq_items', None)
print("\n".join(cfb_data.columns))

season
game_id
game_play_number
pos_team_id
pos_team
def_pos_team_id
def_pos_team
pos_team_score
def_pos_team_score
half
period
down
distance
EPA
wpa
wp_before
wp_after
def_wp_before
def_wp_after
penalty_detail
yds_penalty
penalty_1st_conv
new_series
firstD_by_kickoff
firstD_by_poss
firstD_by_penalty
firstD_by_yards
def_EPA
rz_play
scoring_opp
middle_8
stuffed_run
change_of_pos_team
downs_turnover
pos_score_diff_start
pos_score_pts
home_wp_before
away_wp_before
home_wp_after
away_wp_after
end_of_half
orig_play_type
offense_score_play
defense_score_play
pos_score_diff
change_of_poss
rusher_player_name
yds_rushed
passer_player_name
receiver_player_name
yds_receiving
yds_sacked
sack_players
sack_player_name
sack_player_name2
pass_breakup_player_name
interception_player_name
yds_int_return
fumble_player_name
fumble_forced_player_name
fumble_recovered_player_name
yds_fumble_return
punter_player_name
yds_punted
yds_punt_return
yds_punt_gained
punt_block_player_name
punt_block_return_player_n

In [25]:
# make total yards on the play variable replaced by statYardage
#cfb_data['yds_play'] = cfb_data['yds_rushed'] + cfb_data['yds_receiving'] + cfb_data['yds_penalty'] + cfb_data['yds_sacked']

# net pasing yards = passing yards - sack yards
cfb_data['net_passing_yards'] = cfb_data['yds_receiving'] - cfb_data['yds_sacked']

# third down conversion rate
cfb_data['third_down_conversion'] = np.where(cfb_data['down_3'] == 1, np.where(cfb_data['first_down_created'] == 1, 1, 0), np.nan)

In [27]:
cfb_data_no_na = cfb_data.copy()

In [28]:
cfb_data_no_na = cfb_data_no_na[cfb_data_no_na['pos_team'].notna() & cfb_data_no_na['def_pos_team'].notna()]

In [30]:
cfb_data_no_na

,season,game_id,game_play_number,pos_team_id,pos_team,def_pos_team_id,def_pos_team,pos_team_score,def_pos_team_score,half,...,pointAfterAttempt.text,pointAfterAttempt.abbreviation,pointAfterAttempt.value,extra_point_result,two_point_conv_result,mediaId,game_seconds_remaining,yds_play,net_passing_yards,third_down_conversion
0,2021,401282714,1,158,Nebraska,356,Illinois,0,0,1,...,None,None,NaN,None,None,None,3600.0,NaN,NaN,NaN
1,2021,401282714,2,158,Nebraska,356,Illinois,0,0,1,...,None,None,NaN,None,None,None,3600.0,NaN,NaN,NaN
2,2021,401282714,3,158,Nebraska,356,Illinois,0,0,1,...,None,None,NaN,None,None,None,3595.0,NaN,NaN,NaN
3,2021,401282714,4,158,Nebraska,356,Illinois,0,0,1,...,None,None,NaN,None,None,None,3559.0,NaN,NaN,NaN
4,2021,401282714,5,158,Nebraska,356,Illinois,0,0,1,...,None,None,NaN,None,None,None,3504.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
779899,2025,401769076,170,84,Indiana,2390,Miami,27,21,2,...,None,None,NaN,None,None,None,52.0,NaN,NaN,NaN
779900,2025,401769076,171,84,Indiana,2390,Miami,27,21,2,...,None,None,NaN,None,None,None,51.0,NaN,NaN,NaN
779901,2025,401769076,172,84,Indiana,2390,Miami,27,21,2,...,None,None,NaN,None,None,None,44.0,NaN,NaN,NaN
779902,2025,401769076,173,84,Indiana,2390,Miami,27,21,2,...,None,None,NaN,None,None,None,44.0,NaN,NaN,NaN


In [42]:
cfb_data_no_na['yds_penalty'].dtype

dtype('O')

In [44]:
cfb_data_no_na['yds_penalty'] = cfb_data_no_na['yds_penalty'].astype(float)

In [ ]:
agg_cfb_data = cfb_data_no_na.groupby(['season', 'game_id', 'pos_team', 'pos_team_id']).agg(
    points = ('pos_team_score', 'max'),
    points_allowed = ('def_pos_team_score', 'max'),
    home_Score = ('homeScore', 'max'), # test
    away_Score = ('awayScore', 'max'), # test
    avg_yds_play=('statYardage', 'mean'),
    avg_rush_yds_play=('yds_rushed', 'mean'),
    avg_turnovers_play=('is_turnover', 'mean'),
    avg_net_passing_yards=('net_passing_yards', 'mean'),
    avg_first_downs_play=('first_down_created', 'mean'),
    avg_third_down_conversion=('third_down_conversion', 'mean'),
    avg_penlty_yds_play=('yds_penalty', 'mean'),
    week_num=('week', 'first'),
    home_team=('homeTeamName', 'first'),
    away_team=('awayTeamName', 'first')
    pos_team_wp=('wp_after', 'mean')
).reset_index().sort_values(['season', 'game_id', 'pos_team'], ascending=[True, True, True])

In [49]:
agg_cfb_data.head(10)

,season,game_id,pos_team,pos_team_id,points,points_allowed,home_Score,away_Score,avg_yds_play,avg_rush_yds_play,avg_turnovers_play,avg_net_passing_yards,avg_first_downs_play,avg_third_down_conversion,avg_penlty_yds_play,week_num,home_team,away_team
0,2021,401281942,Alabama,333,44,13,13,44,7.775281,4.216216,0.011236,9.000000,0.247191,0.444444,7.500000,1,Miami,Alabama
1,2021,401281942,Miami,2390,13,44,13,44,5.781609,3.888889,0.034483,4.250000,0.160920,0.428571,6.222222,1,Miami,Alabama
2,2021,401281944,Akron,2006,10,60,60,10,2.807018,3.125000,0.000000,4.666667,0.114035,0.130435,5.000000,1,Auburn,Akron
3,2021,401281944,Auburn,2,60,10,60,10,8.905405,9.852941,0.000000,NaN,0.324324,0.428571,2.500000,1,Auburn,Akron
4,2021,401281945,Florida,57,35,14,35,14,5.768421,8.893617,0.021053,NaN,0.252632,0.500000,6.600000,1,Florida,Florida Atlantic
5,2021,401281945,Florida Atlantic,2226,14,35,35,14,4.522222,4.827586,0.011111,6.000000,0.188889,0.312500,6.600000,1,Florida,Florida Atlantic
6,2021,401281946,Clemson,228,3,10,3,10,4.172840,3.000000,0.012346,6.571429,0.123457,0.277778,8.285714,1,Clemson,Georgia
7,2021,401281946,Georgia,61,10,3,3,10,4.130952,4.300000,0.011905,8.000000,0.154762,0.318182,1.666667,1,Clemson,Georgia
8,2021,401281947,Kentucky,96,45,10,45,10,8.037500,5.500000,0.037500,6.000000,0.225000,0.250000,2.000000,1,Kentucky,UL Monroe
9,2021,401281947,UL Monroe,2433,10,45,45,10,1.943820,1.594595,0.000000,7.166667,0.089888,0.227273,9.800000,1,Kentucky,UL Monroe


In [ ]:
'''agg_cfb_data = cfb_data.groupby(['season', 'game_id', 'homeTeamName', 'awayTeamName']).agg(
    home_score=('homeScore', 'max'),
    week_num=('week', 'first'),
    avg_home_score=('homeScore', 'mean'),
    avg_away_score=('awayScore', 'mean'),
    avg_home_yards=('homeYards', 'mean'),
    avg_away_yards=('awayYards', 'mean'),
)'''

In [ ]:
def parse_time_sec(time_str):
    if pd.isna(time_str):
        return 1800.0
    parts = str(time_str).split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    return 1800.0

def parse_split_eff(eff_str, pos=0):
    if pd.isna(eff_str):
        return 0.0
    parts = str(eff_str).split("-")
    if len(parts) == 2:
        try:
            return float(parts[pos])
        except ValueError:
            return 0.0
    return 0.0

In [ ]:
df = raw_team_stats.copy()
df = df.dropna(subset=["points", "points_allowed"])
df = df[df["points"] != df["points_allowed"]]

df["points"] = df["points"].astype(float)
df["points_allowed"] = df["points_allowed"].astype(float)
df["win"] = (df["points"] > df["points_allowed"]).astype(int)

# Parsing nested string fields
df["td_comp"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 0))
df["td_att"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 1))
df["td_comp_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 0))
df["td_att_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["pen_yds"] = df["total_penalties_yards"].apply(lambda x: parse_split_eff(x, 1))
df["pen_yds_opp"] = df["total_penalties_yards_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["top_sec"] = df["possession_time"].apply(parse_time_sec)
df["top_sec_opp"] = df["possession_time_allowed"].apply(parse_time_sec)

df["third_down_pct"] = np.where(df["td_att"] > 0, df["td_comp"] / df["td_att"], 0.0)
df["third_down_pct_opp"] = np.where(df["td_att_opp"] > 0, df["td_comp_opp"] / df["td_att_opp"], 0.0)

num_cols = ["turnovers", "turnovers_allowed", "net_passing_yards", "net_passing_yards_allowed",
            "rushing_yards", "rushing_yards_allowed", "first_downs", "first_downs_allowed"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)df = raw_team_stats.copy()
df = df.dropna(subset=["points", "points_allowed"])
df = df[df["points"] != df["points_allowed"]]

df["points"] = df["points"].astype(float)
df["points_allowed"] = df["points_allowed"].astype(float)
df["win"] = (df["points"] > df["points_allowed"]).astype(int)

# Parsing nested string fields
df["td_comp"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 0))
df["td_att"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 1))
df["td_comp_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 0))
df["td_att_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["pen_yds"] = df["total_penalties_yards"].apply(lambda x: parse_split_eff(x, 1))
df["pen_yds_opp"] = df["total_penalties_yards_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["top_sec"] = df["possession_time"].apply(parse_time_sec)
df["top_sec_opp"] = df["possession_time_allowed"].apply(parse_time_sec)

df["third_down_pct"] = np.where(df["td_att"] > 0, df["td_comp"] / df["td_att"], 0.0)
df["third_down_pct_opp"] = np.where(df["td_att_opp"] > 0, df["td_comp_opp"] / df["td_att_opp"], 0.0)

num_cols = ["turnovers", "turnovers_allowed", "net_passing_yards", "net_passing_yards_allowed",
            "rushing_yards", "rushing_yards_allowed", "first_downs", "first_downs_allowed"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

In [ ]:
# Computing differentials
df["turnover_margin"] = df["turnovers_allowed"] - df["turnovers"]
df["pass_yards_diff"] = df["net_passing_yards"] - df["net_passing_yards_allowed"]
df["rush_yards_diff"] = df["rushing_yards"] - df["rushing_yards_allowed"]
df["first_downs_diff"] = df["first_downs"] - df["first_downs_allowed"]
df["third_down_pct_diff"] = df["third_down_pct"] - df["third_down_pct_opp"]
df["penalty_yards_diff"] = df["pen_yds"] - df["pen_yds_opp"]
df["top_seconds_diff"] = df["top_sec"] - df["top_sec_opp"]
df["is_home"] = (df["home_away"].str.lower() == "home").astype(int)

# Filter every 2nd row to prevent double-counting full game perspectives
cfb_diff = df.iloc[1::2].reset_index(drop=True)

diff_predictors = [
    "is_home", "turnover_margin", "pass_yards_diff", "rush_yards_diff",
    "first_downs_diff", "third_down_pct_diff", "penalty_yards_diff", "top_seconds_diff"
]

cfb_diff = cfb_diff.dropna(subset=["win"] + diff_predictors)
cfb_diff.head()

In [ ]:
X_diff = cfb_diff[diff_predictors]
y_diff = cfb_diff["win"]
strata_diff = cfb_diff["season"]

X_train, X_test, y_train, y_test = train_test_split(
    X_diff, y_diff, test_size=0.20, random_state=2026, stratify=strata_diff
)

test_indices = X_test.index
test_seasons = cfb_diff.loc[test_indices, "season"]

# Standardizing features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=diff_predictors, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=diff_predictors, index=X_test.index)

# Add constant for statsmodels Logit
X_train_sm = sm.add_constant(X_train_scaled)
X_test_sm = sm.add_constant(X_test_scaled)

In [ ]:
# Fit Logistic Regression Model
logit_model_diff = sm.Logit(y_train, X_train_sm).fit(disp=False)

# Feature Importance Summary Table
summary_df = pd.DataFrame({
    "Predictor": logit_model_diff.params.index,
    "Std Beta": logit_model_diff.params.values,
    "Odds Ratio (1 SD)": np.exp(logit_model_diff.params.values),
    "z-statistic": logit_model_diff.tvalues.values,
    "p-value": logit_model_diff.pvalues.values
})
summary_df = summary_df[summary_df["Predictor"] != "const"].copy()
summary_df["Importance (|z|)"] = summary_df["z-statistic"].abs()
summary_df = summary_df.sort_values(by="Importance (|z|)", ascending=False)

print(summary_df.to_string(index=False))

In [ ]:
# Plot Feature Importance
plt.figure(figsize=(9, 5))
sns.barplot(data=summary_df, x="Importance (|z|)", y="Predictor", palette="mako")
plt.title("FBS Win Probability: Feature Importance Ranking (Differentials)", fontsize=13, fontweight="bold")
plt.xlabel("Feature Importance (|z-score|)")
plt.ylabel("Model Feature")
plt.tight_layout()
plt.show()